In [1]:
import pandas as pd
import boto3
import io
from datetime import datetime
from sagemaker.predictor import Predictor
from sagemaker.serializers import IdentitySerializer
from sagemaker.deserializers import JSONDeserializer

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
%store -r

## Model Monitor set up

In [ ]:
#  Reloading a saved split dataframe from S3 parquet
s3 = boto3.client("s3")

def load_df_from_s3_parquet(s3_key: str):
    obj = s3.get_object(Bucket=bucket, Key=s3_key)
    return pd.read_parquet(io.BytesIO(obj["Body"].read()))


prod_df2 = load_df_from_s3_parquet(mon_base_prefix + "prod_df2.parquet")

In [ ]:
#  Fetching latest deployed SageMaker endpoint dynamically
sm = boto3.client("sagemaker")

endpoints = sm.list_endpoints(SortBy="CreationTime", SortOrder="Descending")["Endpoints"]
latest_endpoint = endpoints[0]["EndpointName"]
print("Using latest endpoint:", latest_endpoint)

Using latest endpoint: pytorch-inference-2026-02-14-01-18-27-568


In [ ]:
#  Reconnecting to latest endpoint
predictor = Predictor(
    endpoint_name=latest_endpoint,
    serializer=IdentitySerializer("application/x-image"),
    deserializer=JSONDeserializer()
)

print("Connected to:", latest_endpoint)

Connected to: pytorch-inference-2026-02-14-01-18-27-568


In [ ]:
#  Enabling Data Capture by creating a NEW EndpointConfig
#  updating existing endpoint to use it
#  Using a compact timestamp to keep it unique
endpoint_name = latest_endpoint

desc_ep = sm.describe_endpoint(EndpointName=endpoint_name)
old_cfg_name = desc_ep["EndpointConfigName"]

desc_cfg = sm.describe_endpoint_config(EndpointConfigName=old_cfg_name)
prod_variants = desc_cfg["ProductionVariants"]

ts = datetime.utcnow().strftime("%y%m%d%H%M%S")  # e.g., 260215203258
new_cfg_name = f"catcls-cap-{ts}"                # short + unique, well under 63

capture_s3_uri = f"s3://{s3_bucket}/cat-landmarks-project/endpoint-data-capture/"

data_capture_config = {
    "EnableCapture": True,
    "InitialSamplingPercentage": 100,
    "DestinationS3Uri": capture_s3_uri,
    "CaptureOptions": [{"CaptureMode": "Input"}, {"CaptureMode": "Output"}],
}

sm.create_endpoint_config(
    EndpointConfigName=new_cfg_name,
    ProductionVariants=prod_variants,
    DataCaptureConfig=data_capture_config
)

print("Created EndpointConfig:", new_cfg_name)

sm.update_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=new_cfg_name
)

print("Updating endpoint to enable Data Capture:", endpoint_name)
print("Capture S3:", capture_s3_uri)


/tmp/ipykernel_250/1930081745.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%y%m%d%H%M%S")  # e.g., 260215203258


Created EndpointConfig: catcls-cap-260215203714
Updating endpoint to enable Data Capture: pytorch-inference-2026-02-14-01-18-27-568
Capture S3: s3://sagemaker-us-east-1-549206572067/cat-landmarks-project/endpoint-data-capture/


In [ ]:
#  Generating fresh production traffic
# Monitoring will capture input + prediction automatically

# pick the correct URI column dynamically
uri_col = "s3_uri" if "s3_uri" in prod_df2.columns else "image_id"

predictor.content_type = "application/x-image"
predictor.accept = "application/json"

for _ in range(10):
    s3_uri = prod_df2[uri_col].sample(1).iloc[0]
    bucket, key = s3_uri.replace("s3://", "").split("/", 1)

    payload = s3.get_object(Bucket=bucket, Key=key)["Body"].read()

    print(predictor.predict(payload))



{'prediction': 1}
{'prediction': 1}
{'prediction': 1}
{'prediction': 1}
{'prediction': 1}
{'prediction': 0}
{'prediction': 0}
{'prediction': 0}
{'prediction': 0}
{'prediction': 1}


### Deleteing endpoint

In [6]:
#  Deleteing real-time inference endpoint


#import boto3

#sm = boto3.client("sagemaker")

#endpoint_name = "pytorch-inference-2026-02-14-01-18-27-568"

#sm.delete_endpoint(EndpointName=endpoint_name)

#print("Endpoint deleted:", endpoint_name)



Endpoint deleted: pytorch-inference-2026-02-14-01-18-27-568
